In [6]:
import torch.nn as nn
import torch
import pandas as pd

In [7]:
data = pd.read_csv("Pokemon.csv")
data.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


## Descripción de la tarea

Trabajaremos con un dataset de la serie "Pokémon" que desglosa las capacidades de cada criatura en atributos cuantitativos. La tarea consiste en utilizar arquitecturas de redes neuronales simples (MLP) para identificar patrones que distingan a los Pokémon Legendarios del resto. Deberán procesar estas variables y entrenar un clasificador que maximice la capacidad predictiva sobre la variable objetivo "Legendary".

Consideraciones:
- Debe entregarlo a más tardar el 29 de mayo a las 18:00 horas.
- Debe ser entregado al correo luis.llanca@uach.cl con el asunto "Tarea-1-MLP", el archivo debe llamarse NG-MLP-Tarea1.ipynb donde NG es el número de grupo. Es importante que el asunto sea exactamente el mismo. También, se les pedirá que se anoten en la plantilla (que se compartirá posteriormente) para una pequeña interrogación.
- Por cada 20 minutos de retraso, se descontará una décima de la nota. 
- Si necesitan ayuda, pueden escribir a los correos luis.llanca@uach.cl, luis.llanca@cenia.cl o escribir al discord de usuario: llanking (tengo una foto de mi gata). PD: prefiero mucho más el discord. 

### Explicación del dataset
Explique el dataset en detalle, incluyendo como mínimo una pequeña descripción de cada columna, el tipo de datos que contiene cada columna, y cualquier información adicional relevante para entender el dataset.

# Explicación del Dataset

El dataset utilizado contiene información de distintos Pokémon y sus estadísticas base. Cada fila representa un Pokémon y cada columna corresponde a una característica específica utilizada para analizar o clasificar al Pokémon.

La variable objetivo del problema es `Legendary`, por lo que se trata de un problema de clasificación binaria utilizando redes neuronales, tal como se explica en el material del curso de Redes Neuronales y Machine Learning de Pablo Huijse.

---

## Descripción de las columnas

| Columna | Tipo de dato | Descripción |
|---|---|---|
| `#` | Entero | Número identificador del Pokémon en la Pokédex. |
| `Name` | Texto | Nombre del Pokémon. |
| `Type 1` | Texto | Tipo principal del Pokémon. |
| `Type 2` | Texto / Nulo | Tipo secundario del Pokémon. Algunos no poseen segundo tipo. |
| `Total` | Entero | Suma total de las estadísticas base. |
| `HP` | Entero | Cantidad de puntos de vida. |
| `Attack` | Entero | Nivel de ataque físico. |
| `Defense` | Entero | Nivel de defensa física. |
| `Sp. Atk` | Entero | Poder de ataque especial. |
| `Sp. Def` | Entero | Defensa contra ataques especiales. |
| `Speed` | Entero | Velocidad del Pokémon. |
| `Generation` | Entero | Generación a la que pertenece el Pokémon. |
| `Legendary` | Booleano | Indica si el Pokémon es legendario (`True`) o no (`False`). |

---

## Información relevante del dataset

- Las columnas de estadísticas (`HP`, `Attack`, `Defense`, etc.) corresponden a variables numéricas utilizadas como entrada del modelo.

- Las columnas `Type 1` y `Type 2` son variables categóricas, por lo que deben transformarse a formato numérico antes de entrenar una red neuronal.

- La columna `Legendary` corresponde a la variable objetivo del problema, por lo que el modelo debe aprender a clasificar entre Pokémon legendarios y no legendarios.

- Existen valores faltantes en `Type 2`, ya que algunos Pokémon tienen solamente un tipo.

- Según el material del curso, es recomendable normalizar las variables numéricas antes del entrenamiento para mejorar el funcionamiento del gradiente descendente y del perceptrón multicapa (MLP).

- El dataset presenta desbalance de clases, debido a que existen muchos menos Pokémon legendarios que normales. El material del curso recomienda considerar métricas adicionales al accuracy y utilizar particiones estratificadas en estos casos.




### Preparación del dataset

Realice una preparación del dataset según lo que considere necesario para el entrenamiento de un modelo de clasificación. Justifique las decisiones que tome en este proceso.

# Preparación del dataset

Antes de entrenar el modelo fue necesario realizar una preparación exhaustiva de los datos para asegurar un entrenamiento robusto y evitar filtraciones de información (*data leakage*):

* Se eliminaron las columnas `Name`, `#` y `Generation`, ya que no aportaban poder predictivo generalizable para el proceso de clasificación.
* La variable objetivo `Legendary` (booleana) se transformó a formato numérico binario (1 y 0) para el cálculo de la función de pérdida.
* Las variables categóricas `Type 1` y `Type 2` fueron transformadas mediante **One-Hot Encoding**, incluyendo una categoría explícita para los valores nulos de `Type 2` para evitar pérdida de datos, debido a que las redes neuronales trabajan con entradas puramente numéricas.
* Se realizó una **partición estratificada** de los datos en conjuntos de entrenamiento y prueba. Esto es necesario debido al desbalance de clases del dataset, garantizando que la baja proporción de Pokémon Legendarios se mantenga constante en ambos conjuntos.
* Las variables numéricas continuas fueron normalizadas utilizando **StandardScaler** para que todas se encontraran en escalas similares, lo que ayuda a la convergencia del modelo mediante gradiente descendente. El ajuste del escalador se realizó exclusivamente sobre el conjunto de entrenamiento para no contaminar el modelo con la distribución estadística de los datos de prueba.
* Los conjuntos resultantes se transformaron a **Tensores flotantes de PyTorch** (`torch.float32`), cumpliendo con la estructura de datos obligatoria para alimentar las capas densas del modelo.

In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Cargar y limpiar columnas no relevantes
df = pd.read_csv("Pokemon.csv")
df = df.drop(columns=["Name", "#", "Generation"])

# 2. Separar características (X) y objetivo (y)
X = df.drop(columns=["Legendary"])
y = df["Legendary"].astype(int) # Convertimos True/False a 1 y 0

# 3. Aplicar One-Hot Encoding a variables categóricas
X = pd.get_dummies(X, columns=["Type 1", "Type 2"], dummy_na=True) # dummy_na maneja los nulos de Type 2

# 4. Dividir los datos en Entrenamiento (Train) y Prueba (Test) de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 5. Escalar SOLO las variables numéricas continuas
num_cols = ['Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
scaler = StandardScaler()

# Ajustamos el escalador solo en Train y lo aplicamos a Train y Test
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# 6. Convertir a Tensores de PyTorch para el MLP
# Convertimos los booleanos del One-Hot a enteros antes de pasarlos a tensores flotantes
X_train_tensor = torch.tensor(X_train.astype(float).values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1) 

X_test_tensor = torch.tensor(X_test.astype(float).values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

### Definición del modelo  
Defina al menos 3 arquitecturas de redes neuronales simples (MLP) para el problema de clasificación. Justifique las decisiones que tome en la definición de cada arquitectura. Las definiciones se deben hacer en un archivo ```models.py``` e importarlas en este cuadernillo. Debe seleccionar "la mejor" arquitectura para el entrenamiento, y justificar su elección.

### Definición de optimizador y función de costo  
Defina un optimizador y una función de costo adecuado para el entrenamiento del modelo. Justifique sus decisiones.

### Entrenamiento del modelo
Entrene el modelo seleccionado utilizando el dataset preparado. Justifique las decisiones que tome en el proceso de entrenamiento, incluyendo la selección de hiperparámetros, el número de épocas, el tamaño del batch, etc.

### Evaluación del modelo
Evalúe el modelo utilizando métricas adecuadas para este problema de clasificación. Justifique la selección de las métricas utilizadas y discuta los resultados obtenidos. 

### Preguntas finales
1. Sobre la matriz de confusión, interprete los resultados obtenidos. Con sus palabras defina que significa cada tipo de error. ¿Elegiría a Pokémon ubicados en FP o FN para su equipo?
2. Busque un caso mal clasificado por el modelo, e interprete por qué cree que el modelo se equivocó en ese caso.
3. ¿Cúal fue el mayor desafío que enfrentó al realizar esta tarea? ¿Cómo lo solucionó?



### IA Generativa

Con el fin de ocupar IA Generativa de manera responsable, se les pide que respondan a las siguientes preguntas:
1. ¿Utilizó alguna herramienta de IA Generativa para realizar esta tarea? En caso afirmativo, indique cuál o cuáles herramientas utilizó.
2. ¿En qué parte o partes de la tarea utilizó estas herramientas?